# Figure 3 — Explorations
Scratchpad cells exploring IFFL parameter sensitivity and switching strategies.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.integrate import solve_ivp
import sys
sys.path.insert(0, '../src')
from rpa_control.style import set_style

set_style()
plt.rcParams['text.usetex'] = False

def clean_ax(ax):
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

PARAMS = dict(alpha=1.0, delta=1.0, beta=1.0, gamma=1.0)

def steady_state(u, p=PARAMS):
    x_ss = p['alpha'] * u / p['delta']
    y_ss = p['beta'] * p['delta'] / (p['gamma'] * p['alpha'])
    return x_ss, y_ss

def iffl_rhs(y_state, u, p=PARAMS):
    x, y = y_state
    dx_dt = p['alpha'] * u - p['delta'] * x
    dy_dt = p['beta'] * u - p['gamma'] * x * y
    return np.array([dx_dt, dy_dt])

PANEL_B = dict(u_base=1.0, u_pert=2.0, t_switch=1.0, t_end=11.0)

In [ ]:
# Parameter sensitivity: vary each of alpha, beta, gamma, delta at 0.1 and 10
# 8 plots in a 2x4 grid, all using PANEL_B timing and u values

param_sweeps = [
    ('alpha', 0.1), ('alpha', 10.0),
    ('beta',  0.1), ('beta',  10.0),
    ('gamma', 0.1), ('gamma', 10.0),
    ('delta', 0.1), ('delta', 10.0),
]

fig, axes = plt.subplots(2, 4, figsize=(14, 5), sharey=False)
axes = axes.flatten()

for ax, (param, val) in zip(axes, param_sweeps):
    p = {**PARAMS, param: val}
    x_ss, y_ss = steady_state(PANEL_B['u_base'], p)

    t1 = np.linspace(0, PANEL_B['t_switch'], 250)
    sol1 = solve_ivp(lambda t_, y, p=p: iffl_rhs(y, PANEL_B['u_base'], p).tolist(),
                     [0, PANEL_B['t_switch']], [x_ss, y_ss],
                     dense_output=True, max_step=0.05)
    y_sw = sol1.sol(PANEL_B['t_switch'])

    t2 = np.linspace(PANEL_B['t_switch'], PANEL_B['t_end'], 500)
    sol2 = solve_ivp(lambda t_, y, p=p: iffl_rhs(y, PANEL_B['u_pert'], p).tolist(),
                     [PANEL_B['t_switch'], PANEL_B['t_end']], y_sw,
                     dense_output=True, max_step=0.05)

    t_full = np.concatenate([t1, t2[1:]])
    y_full = np.concatenate([sol1.sol(t1)[1], sol2.sol(t2[1:])[1]])

    ax.axvspan(PANEL_B['t_switch'], PANEL_B['t_end'], color='#fcba03', alpha=0.15, lw=0)
    ax.plot(t_full, y_full, lw=1.5, color='C0')
    ax.axhline(y_ss, color='black', ls='--', lw=0.8, alpha=0.5)
    ax.set_title(rf'$\{param}={val}$', fontsize=10)
    ax.set_xlabel('Time', fontsize=8)
    ax.set_ylabel('$y(t)$', fontsize=8)
    clean_ax(ax)

plt.suptitle('Parameter sensitivity: each parameter varied independently (others at default = 1)',
             fontsize=10, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Triggered switch: u=1→2 at t=1, then u=2→1 when y peaks
# Peak detected via event: dy/dt crosses zero from above (direction=-1)

T_END = 20.0
T_SWITCH1 = 1.0

x_ss0, y_ss0 = steady_state(1.0)

sol1 = solve_ivp(
    lambda t_, y: iffl_rhs(y, 1.0).tolist(),
    [0, T_SWITCH1], [x_ss0, y_ss0], dense_output=True, max_step=0.05)
y_sw1 = sol1.sol(T_SWITCH1)

def y_peak_event(t_, y):
    return PARAMS['beta'] * 2.0 - PARAMS['gamma'] * y[0] * y[1]
y_peak_event.terminal  = True
y_peak_event.direction = -1

sol2 = solve_ivp(
    lambda t_, y: iffl_rhs(y, 2.0).tolist(),
    [T_SWITCH1, T_END], y_sw1, dense_output=True, max_step=0.02,
    events=y_peak_event)
t_peak = sol2.t_events[0][0]
y_sw2  = sol2.sol(t_peak)

sol3 = solve_ivp(
    lambda t_, y: iffl_rhs(y, 1.0).tolist(),
    [t_peak, T_END], y_sw2, dense_output=True, max_step=0.05)

t1 = np.linspace(0,         T_SWITCH1, 100)
t2 = np.linspace(T_SWITCH1, t_peak,    300)
t3 = np.linspace(t_peak,    T_END,     500)
t_all = np.concatenate([t1, t2[1:], t3[1:]])
y_all = np.concatenate([sol1.sol(t1)[1], sol2.sol(t2[1:])[1], sol3.sol(t3[1:])[1]])

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.axvspan(T_SWITCH1, t_peak, color='#fcba03', alpha=0.15, lw=0, label='$u=2$')
ax.plot(t_all, y_all, lw=1.5, color='C0')
ax.axhline(y_ss0, color='black', ls='--', lw=0.8, alpha=0.5, label=r'$y_{ss}$')
ax.axvline(t_peak, color='C2', ls=':', lw=1.2, label=f'Peak at $t={t_peak:.2f}$')
ax.set_xlabel('Time'); ax.set_ylabel('$y(t)$')
ax.set_title(r'Triggered switch: $u\!:\!1\!\to\!2$ at $t=1$, then $u\!:\!2\!\to\!1$ at $y$ peak')
ax.legend(fontsize=8, loc='upper right')
clean_ax(ax); plt.tight_layout(); plt.show()
print(f"y peaked at t = {t_peak:.3f},  y_peak = {y_sw2[1]:.3f}")

In [ ]:
# Continuous switching: adaptive (dy/dt=0) vs two periodic schedules

T_END_SW  = 30.0
T_SWITCH1 = 1.0
N_MAX_PHASES = 30

x_ss0, y_ss0 = steady_state(1.0)

def run_periodic(period, t_end=T_END_SW, t_first_switch=T_SWITCH1):
    segs_t, segs_y = [], []
    t_cur = 0.0
    y_cur = [x_ss0, y_ss0]
    u_cur = 1.0
    next_switch = t_first_switch
    while t_cur < t_end:
        t_stop = min(next_switch, t_end)
        sol = solve_ivp(lambda t_, y, u=u_cur: iffl_rhs(y, u).tolist(),
                        [t_cur, t_stop], y_cur, dense_output=True, max_step=0.05)
        ts = np.linspace(t_cur, t_stop, max(50, int((t_stop - t_cur) * 100)))
        segs_t.append(ts); segs_y.append(sol.sol(ts)[1])
        y_cur = sol.sol(t_stop); t_cur = t_stop
        if t_cur < t_end:
            u_cur = 2.0 if u_cur == 1.0 else 1.0
            next_switch = t_cur + period
    return np.concatenate(segs_t), np.concatenate(segs_y)

sol0 = solve_ivp(lambda t_, y: iffl_rhs(y, 1.0).tolist(),
                 [0, T_SWITCH1], [x_ss0, y_ss0], dense_output=True, max_step=0.05)
segments = [(np.linspace(0, T_SWITCH1, 100), sol0, 1.0)]
switch_times = [T_SWITCH1]
t_cur = T_SWITCH1; y_cur = sol0.sol(T_SWITCH1); u_cur = 2.0

for _ in range(N_MAX_PHASES):
    if t_cur >= T_END_SW: break
    def dydt_zero(t_, y, u=u_cur):
        return PARAMS['beta'] * u - PARAMS['gamma'] * y[0] * y[1]
    dydt_zero.terminal = True; dydt_zero.direction = 0
    sol = solve_ivp(lambda t_, y, u=u_cur: iffl_rhs(y, u).tolist(),
                    [t_cur, T_END_SW], y_cur, dense_output=True, max_step=0.02, events=dydt_zero)
    t_end_seg = sol.t_events[0][0] if len(sol.t_events[0]) > 0 else T_END_SW
    segments.append((np.linspace(t_cur, t_end_seg, 400), sol, u_cur))
    if len(sol.t_events[0]) == 0: break
    switch_times.append(t_end_seg)
    y_cur = sol.sol(t_end_seg); t_cur = t_end_seg
    u_cur = 2.0 if u_cur == 1.0 else 1.0

t_adapt = np.concatenate([ts for ts, sol, u in segments])
y_adapt = np.concatenate([sol.sol(ts)[1] for ts, sol, u in segments])

PERIOD_FAST = 2.0; PERIOD_SLOW = 6.0
t_fast, y_fast = run_periodic(PERIOD_FAST)
t_slow, y_slow = run_periodic(PERIOD_SLOW)

fig, ax = plt.subplots(figsize=(9, 3.5))
for (ts, sol, u) in segments:
    if u == 2.0:
        ax.axvspan(ts[0], ts[-1], color='#fcba03', alpha=0.12, lw=0)
ax.plot(t_adapt, y_adapt, lw=1.8, color='C0', label='Adaptive ($\\dot{y}=0$)', zorder=3)
ax.plot(t_fast,  y_fast,  lw=1.2, color='C1', ls='--', label=f'Periodic $T={PERIOD_FAST}$')
ax.plot(t_slow,  y_slow,  lw=1.2, color='C2', ls='--', label=f'Periodic $T={PERIOD_SLOW}$')
ax.axhline(y_ss0, color='black', ls=':', lw=0.8, alpha=0.5, label=r'$y_{ss}$')
ax.set_xlabel('Time'); ax.set_ylabel('$y(t)$')
ax.set_title(r'Adaptive vs periodic switching  (yellow $= u\!=\!2$, adaptive)')
ax.legend(fontsize=8, loc='upper right')
clean_ax(ax); plt.tight_layout(); plt.show()
print(f"Adaptive switches: {len(switch_times)-1}")